# 01 — Explore SKAB

Load the open **SKAB** dataset, inspect sensors and anomaly labels, and demonstrate **gap injection** with mask + dt channels.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sentinelai.config import SENSORS
from sentinelai.data.skab import list_scenarios, load_skab
from sentinelai.data.windows import inject_gaps, make_windows, stack_channels

In [ ]:
print("Scenarios:", list_scenarios())
df = load_skab("valve1")
df.head()

In [ ]:
print(f"Rows: {len(df)}, anomaly rate: {df['anomaly'].mean():.3f}")
df[SENSORS + ['anomaly']].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df['Pressure'], label='Pressure', alpha=0.8)
ax.fill_between(range(len(df)), 0, 1, where=df['anomaly'].astype(bool),
                transform=ax.get_xaxis_transform(), color='red', alpha=0.2, label='anomaly')
ax.set_title('SKAB valve1 — Pressure with anomaly regions')
ax.legend()
plt.show()

## Gap injection + shared windowing

Unlike forward-fill-only, `make_windows()` emits **value + mask + dt** so the model and policy layer know when data was imputed.

In [ ]:
gapped = inject_gaps(df)
batch = make_windows(gapped)
print(f"Windows: {len(batch.labels)}, positive rate: {batch.labels.mean():.3f}")
print(f"Coverage range: [{batch.coverage.min():.2f}, {batch.coverage.max():.2f}]")
print(f"Stacked channels shape: {stack_channels(batch).shape}")

In [ ]:
i = batch.coverage.argmin()
sensor = 0
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
axes[0].plot(batch.values[i, sensor], label='filled value')
axes[1].plot(batch.mask[i, sensor], label='mask (1=real)')
axes[2].plot(batch.dt[i, sensor], label='dt (steps since last real)')
for ax in axes:
    ax.legend()
plt.suptitle(f'Lowest-coverage window (coverage={batch.coverage[i]:.2f})')
plt.tight_layout()
plt.show()